# Part 3 — Advanced Modeling: Ensembles, Tuning, and Full ML Pipeline

Builds on Part 2's classification task (`smoker`). This notebook reproduces Part 2's exact preprocessing (same train/test split, same scaler, `random_state=42`) so all results are directly comparable, then trains decision trees, ensembles (Random Forest, Gradient Boosting), tunes a full `Pipeline` with `GridSearchCV`, runs a manual learning curve, and serializes the best model.

See `README.md` for full written interpretation of every result.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## Setup: Reproduce Part 2 Preprocessing\n\nSame encoding, same 80/20 split (`random_state=42`), same `StandardScaler` fit only on training data — this recreates `Xc_train_scaled`, `Xc_test_scaled`, `y_clf_train`, `y_clf_test` exactly as used in Part 2.

In [ ]:
df = pd.read_csv("cleaned_data.csv")
y_clf = (df["smoker"] == "yes").astype(int)

def encode_categoricals(X_raw, onehot_cols):
    X_enc = X_raw.copy()
    mode_val = X_enc["exercise_freq"].mode()[0]
    X_enc["exercise_freq"] = X_enc["exercise_freq"].fillna(mode_val)
    order_map = {"none": 0, "low": 1, "moderate": 2, "high": 3}
    X_enc["exercise_freq"] = X_enc["exercise_freq"].map(order_map)
    X_enc = pd.get_dummies(X_enc, columns=onehot_cols, drop_first=True)
    return X_enc

X_clf_raw = df.drop(columns=["annual_charges", "smoker"]).copy()
X_clf = encode_categoricals(X_clf_raw, onehot_cols=["sex", "region"])
print(f"X_clf columns: {list(X_clf.columns)}")

In [ ]:
Xc_train, Xc_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)
print(f"Train shape: {Xc_train.shape}, Test shape: {Xc_test.shape}")

scaler_clf = StandardScaler()
scaler_clf.fit(Xc_train)
Xc_train_scaled = scaler_clf.transform(Xc_train)
Xc_test_scaled = scaler_clf.transform(Xc_test)

## Task 1: Decision Tree Baseline (Unconstrained)

In [ ]:
tree_unconstrained = DecisionTreeClassifier(random_state=42)
tree_unconstrained.fit(Xc_train_scaled, y_clf_train)

train_acc_unc = accuracy_score(y_clf_train, tree_unconstrained.predict(Xc_train_scaled))
test_acc_unc = accuracy_score(y_clf_test, tree_unconstrained.predict(Xc_test_scaled))
print(f"Unconstrained Tree -> Train accuracy: {train_acc_unc:.4f} | Test accuracy: {test_acc_unc:.4f}")
print(f"Train-test gap: {train_acc_unc - test_acc_unc:.4f}")

## Task 2: Controlled Decision Tree (max_depth=5, min_samples_split=20)

In [ ]:
tree_controlled = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
tree_controlled.fit(Xc_train_scaled, y_clf_train)

train_acc_ctrl = accuracy_score(y_clf_train, tree_controlled.predict(Xc_train_scaled))
test_acc_ctrl = accuracy_score(y_clf_test, tree_controlled.predict(Xc_test_scaled))
print(f"Controlled Tree -> Train accuracy: {train_acc_ctrl:.4f} | Test accuracy: {test_acc_ctrl:.4f}")
print(f"Train-test gap: {train_acc_ctrl - test_acc_ctrl:.4f}")

## Task 3: Gini vs Entropy (max_depth=5)

In [ ]:
tree_gini = DecisionTreeClassifier(max_depth=5, criterion="gini", random_state=42)
tree_gini.fit(Xc_train_scaled, y_clf_train)
test_acc_gini = accuracy_score(y_clf_test, tree_gini.predict(Xc_test_scaled))

tree_entropy = DecisionTreeClassifier(max_depth=5, criterion="entropy", random_state=42)
tree_entropy.fit(Xc_train_scaled, y_clf_train)
test_acc_entropy = accuracy_score(y_clf_test, tree_entropy.predict(Xc_test_scaled))

print(f"Gini (max_depth=5) -> Test accuracy: {test_acc_gini:.4f}")
print(f"Entropy (max_depth=5) -> Test accuracy: {test_acc_entropy:.4f}")

## Task 4: Random Forest (n_estimators=100, max_depth=10)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(Xc_train_scaled, y_clf_train)

train_acc_rf = accuracy_score(y_clf_train, rf.predict(Xc_train_scaled))
test_acc_rf = accuracy_score(y_clf_test, rf.predict(Xc_test_scaled))
rf_proba = rf.predict_proba(Xc_test_scaled)[:, 1]
auc_rf = roc_auc_score(y_clf_test, rf_proba)

print(f"Random Forest -> Train accuracy: {train_acc_rf:.4f} | Test accuracy: {test_acc_rf:.4f} | AUC: {auc_rf:.4f}")

In [ ]:
feature_importance_table = pd.DataFrame({
    "feature": X_clf.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)
print("Top 5 features by importance:")
print(feature_importance_table.head(5))

## Task 4a: Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(Xc_train_scaled, y_clf_train)

train_acc_gb = accuracy_score(y_clf_train, gb.predict(Xc_train_scaled))
test_acc_gb = accuracy_score(y_clf_test, gb.predict(Xc_test_scaled))
gb_proba = gb.predict_proba(Xc_test_scaled)[:, 1]
auc_gb = roc_auc_score(y_clf_test, gb_proba)

print(f"Gradient Boosting -> Train accuracy: {train_acc_gb:.4f} | Test accuracy: {test_acc_gb:.4f} | AUC: {auc_gb:.4f}")

## Task 4b: Feature Ablation Study\n\nUsing the Random Forest's `feature_importances_`, identify the 5 lowest-importance features, remove them, and retrain an identical Random Forest to compare test AUC.

In [ ]:
lowest5 = feature_importance_table.tail(5)
print("5 lowest-importance features:")
print(lowest5)

In [ ]:
cols_to_drop = lowest5["feature"].tolist()
drop_idx = [list(X_clf.columns).index(c) for c in cols_to_drop]
keep_idx = [i for i in range(X_clf.shape[1]) if i not in drop_idx]

Xc_train_reduced = Xc_train_scaled[:, keep_idx]
Xc_test_reduced = Xc_test_scaled[:, keep_idx]

rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reduced.fit(Xc_train_reduced, y_clf_train)
rf_reduced_proba = rf_reduced.predict_proba(Xc_test_reduced)[:, 1]
auc_rf_reduced = roc_auc_score(y_clf_test, rf_reduced_proba)

print(f"Full model (all features) AUC: {auc_rf:.4f}")
print(f"Reduced model (5 lowest-importance features removed) AUC: {auc_rf_reduced:.4f}")
print(f"AUC change: {auc_rf_reduced - auc_rf:+.4f}")

## Task 5: Cross-Validated Comparison

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_reg_cv_model = LogisticRegression(max_iter=1000, class_weight="balanced")
models_for_cv = {
    "Logistic Regression": log_reg_cv_model,
    "Decision Tree (max_depth=5)": DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
}

cv_results = {}
for name, model in models_for_cv.items():
    scores = cross_val_score(model, Xc_train_scaled, y_clf_train, cv=cv, scoring="roc_auc")
    cv_results[name] = (scores.mean(), scores.std())
    print(f"{name}: mean AUC = {scores.mean():.4f} | std AUC = {scores.std():.4f}")

## Task 6: GridSearchCV Hyperparameter Tuning

Built as a `Pipeline` (`SimpleImputer` -> `StandardScaler` -> `RandomForestClassifier`) so the same object handles preprocessing and modeling together. Fit on **unscaled** `Xc_train` since the pipeline does its own scaling internally.

In [ ]:
pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

param_grid = {
    "randomforestclassifier__n_estimators": [50, 100, 200],
    "randomforestclassifier__max_depth": [5, 10, None],
    "randomforestclassifier__min_samples_leaf": [1, 5],
}

n_configs = 1
for v in param_grid.values():
    n_configs *= len(v)
print(f"Total hyperparameter configurations: {n_configs} (x 5 folds = {n_configs * 5} total fits)")

In [ ]:
grid_search = GridSearchCV(
    pipeline, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1
)
grid_search.fit(Xc_train, y_clf_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV score (mean AUC): {grid_search.best_score_:.4f}")

In [ ]:
best_pipeline = grid_search.best_estimator_
best_pipeline_test_auc = roc_auc_score(y_clf_test, best_pipeline.predict_proba(Xc_test)[:, 1])
print(f"Best pipeline test-set AUC: {best_pipeline_test_auc:.4f}")

## Task 7: Manual Learning Curve\n\nRefit the best pipeline's hyperparameters on progressively larger subsets of the training data (first 20%, 40%, ..., 100% of rows) to see how training and test AUC evolve.

In [ ]:
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
learning_curve_rows = []

Xc_train_reset = Xc_train.reset_index(drop=True)
y_clf_train_reset = y_clf_train.reset_index(drop=True)

for f in fractions:
    n_rows = int(f * len(Xc_train_reset))
    X_sub = Xc_train_reset.iloc[:n_rows]
    y_sub = y_clf_train_reset.iloc[:n_rows]

    pipeline_f = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        RandomForestClassifier(random_state=42, **{
            k.split("__")[1]: v for k, v in grid_search.best_params_.items()
        })
    )
    pipeline_f.fit(X_sub, y_sub)

    train_auc = roc_auc_score(y_sub, pipeline_f.predict_proba(X_sub)[:, 1])
    test_auc = roc_auc_score(y_clf_test, pipeline_f.predict_proba(Xc_test)[:, 1])
    learning_curve_rows.append({
        "Training fraction": f, "n_rows": n_rows, "Training AUC": train_auc, "Test AUC": test_auc
    })

learning_curve_table = pd.DataFrame(learning_curve_rows)
print(learning_curve_table)

## Task 8: Serialize the Best Model

In [ ]:
joblib.dump(best_pipeline, "best_model.pkl")
print("Saved best_model.pkl")

### Reload and Predict (sanity check)

In [ ]:
loaded_model = joblib.load("best_model.pkl")

hand_crafted_rows = pd.DataFrame([
    {  # Row A: young, low BMI, no kids, high exercise, no prior conditions
        "age": 25, "bmi": 22.0, "children": 0, "exercise_freq": 3, "prior_conditions": 0,
        "sex_male": 1, "region_northwest": 0, "region_southeast": 0, "region_southwest": 0,
    },
    {  # Row B: older, high BMI, several kids, no exercise, multiple prior conditions
        "age": 58, "bmi": 34.5, "children": 3, "exercise_freq": 0, "prior_conditions": 3,
        "sex_male": 0, "region_northwest": 0, "region_southeast": 1, "region_southwest": 0,
    },
])
hand_crafted_rows = hand_crafted_rows[X_clf.columns]

predictions = loaded_model.predict(hand_crafted_rows)
probabilities = loaded_model.predict_proba(hand_crafted_rows)[:, 1]
for i, (pred, proba) in enumerate(zip(predictions, probabilities)):
    print(f"Row {chr(65+i)}: predicted class = {pred} (P(smoker) = {proba:.4f})")

## Summary Comparison Table (All Models, Parts 2 & 3)

In [ ]:
test_aucs = {}
for name, model in models_for_cv.items():
    model.fit(Xc_train_scaled, y_clf_train)
    proba = model.predict_proba(Xc_test_scaled)[:, 1]
    test_aucs[name] = roc_auc_score(y_clf_test, proba)

test_aucs["Tuned RF (GridSearchCV pipeline)"] = best_pipeline_test_auc
cv_results["Tuned RF (GridSearchCV pipeline)"] = (grid_search.best_score_, np.nan)

summary_rows = []
for name in list(models_for_cv.keys()) + ["Tuned RF (GridSearchCV pipeline)"]:
    mean_auc, std_auc = cv_results[name]
    summary_rows.append({
        "Model": name,
        "CV Mean AUC": mean_auc,
        "CV Std AUC": std_auc,
        "Test AUC": test_aucs[name],
    })

summary_table = pd.DataFrame(summary_rows)
print(summary_table)